In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import random
from collections import defaultdict
import os

# Enable memory-efficient Scaled Dot-Product Attention (SDPA) if using PyTorch 2.0+
if torch.cuda.is_available():
    try:
        import packaging.version as V
        if V.Version(torch.__version__) >= V.Version("2.0"):
            torch.backends.cuda.enable_mem_efficient_sdp(True)
            print("Enabled Scaled Dot-Product Attention (SDPA).")
    except Exception:
        pass

Enabled Scaled Dot-Product Attention (SDPA).


In [ ]:
class EdgeAwareLoRALinear(nn.Module):
    """
    Custom PyTorch module implementing a Mixture of Experts (MoE) LoRA.
    It replaces a standard linear layer, using a router to select expert pathways.
    """
    def __init__(self, original_layer, num_experts=4, top_k=2, shared_rank=4, expert_hidden_dim=32, lora_alpha=4, dropout=0.1):
        super().__init__()

        self.num_experts = num_experts
        self.top_k = top_k
        self.shared_rank = shared_rank
        self.expert_hidden_dim = expert_hidden_dim
        self.lora_alpha = lora_alpha
        self.scaling = self.lora_alpha / self.shared_rank
        self.original_layer = original_layer

        in_features = original_layer.in_features
        out_features = original_layer.out_features

        self.router = nn.Linear(in_features, self.num_experts, bias=False)
        self.lora_A_shared = nn.Linear(in_features, self.shared_rank, bias=False)

        self.lora_B_experts = nn.ModuleDict()
        self.lora_A_experts = nn.ModuleDict()

        for i in range(self.num_experts):
            expert_name = str(i)
            self.lora_B_experts[expert_name] = nn.Linear(self.shared_rank, self.expert_hidden_dim, bias=False)
            self.lora_A_experts[expert_name] = nn.Linear(self.expert_hidden_dim, self.shared_rank, bias=False)
            nn.init.kaiming_uniform_(self.lora_B_experts[expert_name].weight, a=math.sqrt(5))
            nn.init.kaiming_uniform_(self.lora_A_experts[expert_name].weight, a=math.sqrt(5))

        self.lora_B_shared = nn.Linear(self.shared_rank, out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_A_shared.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B_shared.weight)
        self.lora_dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, sequence_length, hidden_dim = x.shape
        original_output = self.original_layer(x)
        x_flat = x.reshape(-1, hidden_dim)

        router_logits = self.router(x_flat)
        routing_weights_before = F.softmax(router_logits, dim=1, dtype=torch.float)
        routing_weights, selected_experts = torch.topk(routing_weights_before, self.top_k, dim=-1)
        routing_weights /= routing_weights.sum(dim=-1, keepdim=True)
        routing_weights = routing_weights.to(x_flat.dtype)
        expert_mask = F.one_hot(selected_experts, num_classes=self.num_experts).permute(2, 1, 0)

        x_lora = self.lora_A_shared(self.lora_dropout(x_flat))
        combined_expert_output = torch.zeros(
            (batch_size * sequence_length, self.shared_rank), dtype=x_lora.dtype, device=x_lora.device
        )

        for expert_idx in range(self.num_experts):
            idx, top_x = torch.where(expert_mask[expert_idx])
            if top_x.shape[0] == 0:
                continue
            expert_input = x_lora[top_x]
            expert_output = self.lora_A_experts[str(expert_idx)](self.lora_B_experts[str(expert_idx)](expert_input))
            current_expert_output = expert_output * routing_weights[top_x, idx, None]
            combined_expert_output.index_add_(0, top_x, current_expert_output.to(x_lora.dtype))

        final_lora_output = self.lora_B_shared(combined_expert_output)
        final_lora_output = final_lora_output.reshape(batch_size, sequence_length, -1)
        final_lora_output = final_lora_output * self.scaling

        if self.training:
            P = routing_weights_before
            imp = P.mean(dim=0)
            with torch.no_grad():
                assign_counts = torch.bincount(
                    selected_experts.reshape(-1), minlength=self.num_experts
                ).float()
                load = assign_counts / assign_counts.sum().clamp_min(1.0)
            self._lb_loss = self.num_experts * (imp * load).sum()
        else:
            self._lb_loss = None

        return original_output + final_lora_output

In [ ]:
class LBTrainer(Trainer):
    """ Custom Trainer to incorporate the load balancing loss. """
    def __init__(self, *args, lb_coef: float = 1e-2, **kwargs):
        super().__init__(*args, **kwargs)
        self.lb_coef = lb_coef

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # Pass kwargs to the parent class to handle new arguments
        loss, outputs = super().compute_loss(model, inputs, return_outputs=True, **kwargs)
        aux_loss = None
        for m in model.modules():
            if isinstance(m, EdgeAwareLoRALinear):
                lb = getattr(m, "_lb_loss", None)
                if lb is not None:
                    aux_loss = lb if aux_loss is None else (aux_loss + lb)
        if aux_loss is not None:
            loss = loss + self.lb_coef * aux_loss
        return (loss, outputs) if return_outputs else loss

In [ ]:
def patch_roberta_with_edge_aware_lora(model, **kwargs):
    """ Replaces target linear layers in RoBERTa with our custom MoE LoRA layer. """
    for layer in model.encoder.layer:
        layer.attention.self.query = EdgeAwareLoRALinear(layer.attention.self.query, **kwargs)
        layer.attention.self.value = EdgeAwareLoRALinear(layer.attention.self.value, **kwargs)
    return model


def compute_metrics(eval_pred):
    """ Calculates evaluation metrics (accuracy, F1, etc.). """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

In [ ]:
print("Loading and preparing dataset...")
dataset = load_dataset("glue", "mnli")
tokenizer = RobertaTokenizer.from_pretrained("roberta-large")

def tokenize(example):
    return tokenizer(example["premise"], example["hypothesis"], truncation=True, padding="max_length", max_length=128)

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

train_dataset = dataset["train"]
eval_dataset = dataset["validation_matched"]
print("Dataset preparation finished.")

Loading and preparing dataset...
Dataset preparation finished.


In [ ]:
seeds = [42, 123, 7, 99, 101]
all_results = []

for seed in seeds:
    print(f"\n{'='*30}\n  STARTING RUN FOR SEED: {seed}\n{'='*30}\n")

    set_seed(seed)
    out_dir = f"./ourlora_results/seed{seed}"
    os.makedirs(out_dir, exist_ok=True)

    # --- Model Initialization (Fresh model for each seed) ---
    model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=3)
    model.roberta = patch_roberta_with_edge_aware_lora(
        model.roberta, num_experts=4, top_k=2, shared_rank=4, expert_hidden_dim=32
    )

    # Freeze all parameters except the new LoRA modules
    trainable_modules = ["router", "lora_A_shared", "lora_B_shared", "lora_B_experts", "lora_A_experts"]
    for name, param in model.named_parameters():
        if not any(trainable_module in name for trainable_module in trainable_modules):
            param.requires_grad = False

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Seed {seed} | Trainable Params: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")

    # --- Training Arguments ---
    training_args = TrainingArguments(
        output_dir=f"./edge-aware-lora-roberta-mnli-seed-{seed}",
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=3e-4,
        per_device_train_batch_size=128,
        per_device_eval_batch_size=128,
        num_train_epochs=10,
        weight_decay=0.01,
        warmup_ratio=0.06,
        lr_scheduler_type="linear",
        logging_dir=f"./logs/seed-{seed}",
        logging_steps=500,
        report_to="none",
        fp16=True,
        seed=seed,
    )

    # --- Trainer Initialization and Training ---
    trainer = LBTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        lb_coef=5e-3,
    )

    print(f"Starting training for seed {seed}...")
    trainer.train()
    print(f"Training finished for seed {seed}.")

    # --- Post-Training Analysis & Saving ---
    print(f"\n--- Analysis and Saving for SEED: {seed} ---")
    model.save_pretrained(out_dir)

    # Generate and save learning curve plots
    log_history_df = pd.DataFrame(trainer.state.log_history)
    train_logs = log_history_df.dropna(subset=['loss'])
    eval_logs = log_history_df.dropna(subset=['eval_loss'])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle(f"Training Metrics for Seed {seed}", fontsize=16)

    ax1.plot(train_logs['step'], train_logs['loss'], label='Training Loss')
    ax1.plot(eval_logs['step'], eval_logs['eval_loss'], label='Validation Loss')
    ax1.set_title('Training & Validation Loss')
    ax1.set_xlabel('Steps'); ax1.set_ylabel('Loss')
    ax1.legend(); ax1.grid(True)

    ax2.plot(eval_logs['step'], eval_logs['eval_accuracy'], label='Validation Accuracy', color='green')
    ax2.set_title('Validation Accuracy')
    ax2.set_xlabel('Steps'); ax2.set_ylabel('Accuracy')
    ax2.legend(); ax2.grid(True)

    plt.tight_layout()
    plot_path = f"{out_dir}/learning_curve.png"
    plt.savefig(plot_path)
    plt.close()
    print(f"Learning curve plot saved to: {plot_path}")

    # Aggregate and save metrics to CSV
    # ... (code for creating combined_df is here) ...
    csv_path = f"{out_dir}/epoch_metrics.csv"
    # combined_df.to_csv(csv_path, index=False, float_format="%.4f")
    # For simplicity, we just save the full log history.
    log_history_df.to_csv(csv_path, index=False, float_format="%.4f")


    print(f"Results for seed {seed} saved to {out_dir}")
    print(f"\n{'='*30}\n  FINISHED RUN FOR SEED: {seed}\n{'='*30}\n")

    # Store the final eval accuracy
    final_eval_metrics = {k: v for k, v in trainer.state.log_history[-1].items() if k.startswith('eval')}
    final_eval_metrics['seed'] = seed
    all_results.append(final_eval_metrics)


  STARTING RUN FOR SEED: 42



Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Seed 42 | Trainable Params: 638,976 / 356,001,795 (0.18%)
Starting training for seed 42...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.621800,0.321206,0.880591,0.881836,0.880591,0.881016
2,0.575000,0.295355,0.895364,0.895969,0.895364,0.895520
3,0.548400,0.294049,0.891798,0.895877,0.891798,0.892668
4,0.537400,0.270152,0.901172,0.901843,0.901172,0.901396
5,0.526100,0.286798,0.899032,0.900790,0.899032,0.899431
6,0.518800,0.275604,0.901375,0.901750,0.901375,0.901472
7,0.505200,0.276633,0.901885,0.902690,0.901885,0.902100
8,0.503000,0.275005,0.902700,0.903499,0.902700,0.902964
9,0.491400,0.278676,0.903107,0.903938,0.903107,0.903346
10,0.489000,0.279308,0.902292,0.903284,0.902292,0.902584


Training finished for seed 42.

--- Analysis and Saving for SEED: 42 ---
Learning curve plot saved to: ./ourlora_results/seed42/learning_curve.png
Results for seed 42 saved to ./ourlora_results/seed42

  FINISHED RUN FOR SEED: 42


  STARTING RUN FOR SEED: 123



...

Training finished for seed 99.

--- Analysis and Saving for SEED: 99 ---
Learning curve plot saved to: ./ourlora_results/seed99/learning_curve.png
Results for seed 99 saved to ./ourlora_results/seed99

  FINISHED RUN FOR SEED: 99


  STARTING RUN FOR SEED: 101



Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Seed 101 | Trainable Params: 638,976 / 356,001,795 (0.18%)
Starting training for seed 101...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.594100,0.290408,0.895059,0.895592,0.895059,0.895188
2,0.562200,0.266916,0.902394,0.902033,0.902394,0.902104
3,0.543000,0.265648,0.903006,0.903252,0.903006,0.903097
4,0.526800,0.267901,0.903617,0.904699,0.903617,0.903947
5,0.518900,0.273042,0.903923,0.904823,0.903923,0.904217
6,0.505600,0.263135,0.905349,0.905185,0.905349,0.905242
7,0.500200,0.271180,0.906673,0.907021,0.906673,0.906716
8,0.490100,0.278513,0.903617,0.905175,0.903617,0.904029
9,0.492100,0.277367,0.905756,0.906359,0.905756,0.905924
10,0.481500,0.275478,0.907489,0.907814,0.907489,0.907572


Training finished for seed 101.

--- Analysis and Saving for SEED: 101 ---
Learning curve plot saved to: ./ourlora_results/seed101/learning_curve.png
Results for seed 101 saved to ./ourlora_results/seed101

  FINISHED RUN FOR SEED: 101



In [ ]:
print("\n\n--- ALL SEED RUNS COMPLETED ---")
results_df = pd.DataFrame(all_results)
print("Summary of final evaluation accuracy across all seeds:")
print(results_df[['seed', 'eval_accuracy', 'eval_loss']])

print("\nAverage validation accuracy: {:.4f}".format(results_df['eval_accuracy'].mean()))
print("Standard deviation of validation accuracy: {:.4f}".format(results_df['eval_accuracy'].std()))
print("Peak validation accuracy:    {:.4f}".format(results_df['eval_accuracy'].max()))
print("\nExperiment finished successfully! 🚀")



--- ALL SEED RUNS COMPLETED ---
Summary of final evaluation accuracy across all seeds:


KeyError: "['eval_accuracy', 'eval_loss'] not in index"

# OurLoRA with aux for sst-2 roberta-large

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import random
from collections import defaultdict
import os

# Enable memory-efficient Scaled Dot-Product Attention (SDPA) if using PyTorch 2.0+
if torch.cuda.is_available():
    try:
        import packaging.version as V
        if V.Version(torch.__version__) >= V.Version("2.0"):
            torch.backends.cuda.enable_mem_efficient_sdp(True)
            print("Enabled Scaled Dot-Product Attention (SDPA).")
    except Exception:
        pass

Enabled Scaled Dot-Product Attention (SDPA).


In [ ]:
class EdgeAwareLoRALinear(nn.Module):
    """
    Custom PyTorch module implementing a Mixture of Experts (MoE) LoRA.
    It replaces a standard linear layer, using a router to select expert pathways.
    """
    def __init__(self, original_layer, num_experts=4, top_k=2, shared_rank=4, expert_hidden_dim=32, lora_alpha=4, dropout=0.1):
        super().__init__()

        self.num_experts = num_experts
        self.top_k = top_k
        self.shared_rank = shared_rank
        self.expert_hidden_dim = expert_hidden_dim
        self.lora_alpha = lora_alpha
        self.scaling = self.lora_alpha / self.shared_rank
        self.original_layer = original_layer
        self._lb_loss = None # Initialize lb_loss attribute

        in_features = original_layer.in_features
        out_features = original_layer.out_features

        self.router = nn.Linear(in_features, self.num_experts, bias=False)
        self.lora_A_shared = nn.Linear(in_features, self.shared_rank, bias=False)

        self.lora_B_experts = nn.ModuleDict()
        self.lora_A_experts = nn.ModuleDict()

        for i in range(self.num_experts):
            expert_name = str(i)
            self.lora_B_experts[expert_name] = nn.Linear(self.shared_rank, self.expert_hidden_dim, bias=False)
            self.lora_A_experts[expert_name] = nn.Linear(self.expert_hidden_dim, self.shared_rank, bias=False)
            nn.init.kaiming_uniform_(self.lora_B_experts[expert_name].weight, a=math.sqrt(5))
            nn.init.kaiming_uniform_(self.lora_A_experts[expert_name].weight, a=math.sqrt(5))

        self.lora_B_shared = nn.Linear(self.shared_rank, out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_A_shared.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B_shared.weight)
        self.lora_dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, sequence_length, hidden_dim = x.shape
        original_output = self.original_layer(x)
        x_flat = x.reshape(-1, hidden_dim)

        router_logits = self.router(x_flat)
        routing_weights_before = F.softmax(router_logits, dim=1, dtype=torch.float)
        routing_weights, selected_experts = torch.topk(routing_weights_before, self.top_k, dim=-1)
        routing_weights /= routing_weights.sum(dim=-1, keepdim=True)
        routing_weights = routing_weights.to(x_flat.dtype)
        expert_mask = F.one_hot(selected_experts, num_classes=self.num_experts).permute(2, 1, 0)

        x_lora = self.lora_A_shared(self.lora_dropout(x_flat))
        combined_expert_output = torch.zeros(
            (batch_size * sequence_length, self.shared_rank), dtype=x_lora.dtype, device=x_lora.device
        )

        for expert_idx in range(self.num_experts):
            idx, top_x = torch.where(expert_mask[expert_idx])
            if top_x.shape[0] == 0:
                continue
            expert_input = x_lora[top_x]
            expert_output = self.lora_A_experts[str(expert_idx)](self.lora_B_experts[str(expert_idx)](expert_input))
            current_expert_output = expert_output * routing_weights[top_x, idx, None]
            combined_expert_output.index_add_(0, top_x, current_expert_output.to(x_lora.dtype))

        final_lora_output = self.lora_B_shared(combined_expert_output)
        final_lora_output = final_lora_output.reshape(batch_size, sequence_length, -1)
        final_lora_output = final_lora_output * self.scaling

        if self.training:
            P = routing_weights_before
            imp = P.mean(dim=0)
            with torch.no_grad():
                assign_counts = torch.bincount(
                    selected_experts.reshape(-1), minlength=self.num_experts
                ).float()
                load = assign_counts / assign_counts.sum().clamp_min(1.0)
            self._lb_loss = self.num_experts * (imp * load).sum()
        else:
            self._lb_loss = None

        return original_output + final_lora_output

In [ ]:
class LBTrainer(Trainer):
    """ Custom Trainer to incorporate the load balancing loss. """
    def __init__(self, *args, lb_coef: float = 1e-2, **kwargs):
        super().__init__(*args, **kwargs)
        self.lb_coef = lb_coef

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        loss, outputs = super().compute_loss(model, inputs, return_outputs=True, **kwargs)
        aux_loss = None
        for m in model.modules():
            if hasattr(m, "_lb_loss"):
                lb = getattr(m, "_lb_loss", None)
                if lb is not None:
                    aux_loss = lb if aux_loss is None else (aux_loss + lb)
        if aux_loss is not None and self.is_in_train:
            loss = loss + self.lb_coef * aux_loss
        return (loss, outputs) if return_outputs else loss

In [ ]:
def patch_roberta_with_edge_aware_lora(model, **kwargs):
    """ Replaces target linear layers in RoBERTa with our custom MoE LoRA layer. """
    for layer in model.encoder.layer:
        layer.attention.self.query = EdgeAwareLoRALinear(layer.attention.self.query, **kwargs)
        layer.attention.self.value = EdgeAwareLoRALinear(layer.attention.self.value, **kwargs)
    return model

def compute_metrics(eval_pred):
    """ Calculates evaluation metrics (accuracy, F1, etc.). """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

In [ ]:
print("Loading and preparing SST-2 dataset...")
dataset = load_dataset("glue", "sst2")
tokenizer = RobertaTokenizer.from_pretrained("roberta-large")

def tokenize(example):
    return tokenizer(example["sentence"], truncation=True, padding="max_length", max_length=128)

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
train_dataset = dataset["train"]
eval_dataset = dataset["validation"]
print("Dataset preparation finished.")

Loading and preparing SST-2 dataset...
Dataset preparation finished.


In [ ]:
seeds = [42, 123, 7, 99, 101]
lb_coef = 1e-2
all_results = []

In [ ]:
for seed in seeds:
    print(f"\n{'='*40}\n  STARTING RUN: SEED={seed}, LB_COEF={lb_coef}\n{'='*40}\n")

    set_seed(seed)
    # --- CHANGE 2: Simplified output directory ---
    out_dir = f"./ourlora_sst2_results/seed{seed}"
    os.makedirs(out_dir, exist_ok=True)

    model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=2)
    model.roberta = patch_roberta_with_edge_aware_lora(
        model.roberta, num_experts=4, top_k=2, shared_rank=4, expert_hidden_dim=32
    )

    trainable_modules = ["router", "lora_A_shared", "lora_B_shared", "lora_B_experts", "lora_A_experts"]
    for name, param in model.named_parameters():
        if not any(trainable_module in name for trainable_module in trainable_modules):
            param.requires_grad = False

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable Params: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")

    training_args = TrainingArguments(
        output_dir=f"./run_sst2_results/seed{seed}",
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=3e-4,
        per_device_train_batch_size=128,
        per_device_eval_batch_size=128,
        num_train_epochs=10,
        weight_decay=0.01,
        warmup_ratio=0.06,
        lr_scheduler_type="linear",
        logging_dir=f"./logs_sst2/seed{seed}",
        logging_steps=50,
        report_to="none",
        fp16=True,
        seed=seed,
    )

    trainer = LBTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        lb_coef=lb_coef,
    )

    print(f"Starting training...")
    trainer.train()
    print(f"Training finished.")

    print(f"\n--- Analysis and Saving for SEED={seed} ---")

    log_history_df = pd.DataFrame(trainer.state.log_history)
    log_history_df.to_csv(f"{out_dir}/log_history.csv", index=False)

    eval_logs = log_history_df.dropna(subset=['eval_loss'])

    if not eval_logs.empty:
        final_eval_metrics = eval_logs.iloc[-1].to_dict()
        final_eval_metrics['seed'] = seed
        all_results.append(final_eval_metrics)
    else:
        print("No evaluation logs found for this run.")

    print(f"Results for this run saved to {out_dir}")


  STARTING RUN: SEED=42, LB_COEF=0.01



Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable Params: 638,976 / 356,000,770 (0.18%)
Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.679000,0.146538,0.947248,0.947276,0.947248,0.947242
2,0.654400,0.135963,0.955275,0.955276,0.955275,0.955274
3,0.641200,0.137106,0.955275,0.955302,0.955275,0.955278
4,0.627200,0.135285,0.955275,0.955302,0.955275,0.955278
5,0.622200,0.147199,0.954128,0.954159,0.954128,0.954124
6,0.616100,0.148858,0.951835,0.951840,0.951835,0.951833
7,0.609500,0.141017,0.955275,0.955302,0.955275,0.955278
8,0.597500,0.142672,0.954128,0.954128,0.954128,0.954128
9,0.590600,0.149565,0.957569,0.957569,0.957569,0.957568
10,0.586900,0.148151,0.959862,0.959863,0.959862,0.959861


Training finished.

--- Analysis and Saving for SEED=42 ---
Results for this run saved to ./ourlora_sst2_results/seed42

  STARTING RUN: SEED=123, LB_COEF=0.01



...

Training finished.

--- Analysis and Saving for SEED=99 ---
Results for this run saved to ./ourlora_sst2_results/seed99

  STARTING RUN: SEED=101, LB_COEF=0.01



Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable Params: 638,976 / 356,000,770 (0.18%)
Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.672000,0.151517,0.948394,0.948464,0.948394,0.948398
2,0.649100,0.129700,0.954128,0.954159,0.954128,0.954124
3,0.630800,0.145172,0.950688,0.950693,0.950688,0.950689
4,0.620900,0.134079,0.955275,0.955276,0.955275,0.955274
5,0.623400,0.137970,0.955275,0.955345,0.955275,0.955278
6,0.598600,0.140622,0.955275,0.955302,0.955275,0.955278
7,0.593900,0.145251,0.957569,0.957596,0.957569,0.957571
8,0.590500,0.144017,0.958716,0.958729,0.958716,0.958717
9,0.581600,0.139805,0.958716,0.958716,0.958716,0.958716
10,0.581800,0.145267,0.957569,0.957573,0.957569,0.957570


Training finished.

--- Analysis and Saving for SEED=101 ---
Results for this run saved to ./ourlora_sst2_results/seed101


In [ ]:
print("\n\n--- ALL 5 SEED RUNS COMPLETED ---")
if all_results:
    results_df = pd.DataFrame(all_results)
    print("--- Final Evaluation Results per Seed ---")
    # Display accuracy with more precision
    print(results_df[['seed', 'eval_accuracy', 'eval_loss']].to_string(float_format="%.4f"))

    # --- CHANGE 3: Calculate and print average, best, and std dev ---
    avg_accuracy = results_df['eval_accuracy'].mean()
    best_accuracy = results_df['eval_accuracy'].max()
    std_dev_accuracy = results_df['eval_accuracy'].std()

    print("\n--- Summary Across 5 Seeds ---")
    print(f"📊 Average Accuracy: {avg_accuracy:.4f}")
    print(f"🏆 Best Accuracy:    {best_accuracy:.4f}")
    print(f"📈 Std Deviation:    {std_dev_accuracy:.4f}")
else:
    print("No results to summarize.")

print("\nExperiment finished successfully! 🚀")



--- ALL 5 SEED RUNS COMPLETED ---
--- Final Evaluation Results per Seed ---
   seed  eval_accuracy  eval_loss
0    42         0.9599     0.1482
1   123         0.9530     0.1445
2     7         0.9553     0.1510
3    99         0.9622     0.1334
4   101         0.9576     0.1453

--- Summary Across 5 Seeds ---
📊 Average Accuracy: 0.9576
🏆 Best Accuracy:    0.9622
📈 Std Deviation:    0.0036

Experiment finished successfully! 🚀


# OurLoRA with aux for mrpc roberta-large

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import os

# Enable memory-efficient Scaled Dot-Product Attention (SDPA) if using PyTorch 2.0+
if torch.cuda.is_available():
    try:
        import packaging.version as V
        if V.Version(torch.__version__) >= V.Version("2.0"):
            torch.backends.cuda.enable_mem_efficient_sdp(True)
            print("Enabled Scaled Dot-Product Attention (SDPA).")
    except Exception:
        pass

Enabled Scaled Dot-Product Attention (SDPA).


In [ ]:
class EdgeAwareLoRALinear(nn.Module):
    """
    Custom PyTorch module implementing a Mixture of Experts (MoE) LoRA.
    It replaces a standard linear layer, using a router to select expert pathways.
    """
    def __init__(self, original_layer, num_experts=4, top_k=2, shared_rank=4, expert_hidden_dim=32, lora_alpha=4, dropout=0.1):
        super().__init__()

        self.num_experts = num_experts
        self.top_k = top_k
        self.shared_rank = shared_rank
        self.expert_hidden_dim = expert_hidden_dim
        self.lora_alpha = lora_alpha
        self.scaling = self.lora_alpha / self.shared_rank
        self.original_layer = original_layer
        self._lb_loss = None # Initialize lb_loss attribute

        in_features = original_layer.in_features
        out_features = original_layer.out_features

        self.router = nn.Linear(in_features, self.num_experts, bias=False)
        self.lora_A_shared = nn.Linear(in_features, self.shared_rank, bias=False)

        self.lora_B_experts = nn.ModuleDict()
        self.lora_A_experts = nn.ModuleDict()

        for i in range(self.num_experts):
            expert_name = str(i)
            self.lora_B_experts[expert_name] = nn.Linear(self.shared_rank, self.expert_hidden_dim, bias=False)
            self.lora_A_experts[expert_name] = nn.Linear(self.expert_hidden_dim, self.shared_rank, bias=False)
            nn.init.kaiming_uniform_(self.lora_B_experts[expert_name].weight, a=math.sqrt(5))
            nn.init.kaiming_uniform_(self.lora_A_experts[expert_name].weight, a=math.sqrt(5))

        self.lora_B_shared = nn.Linear(self.shared_rank, out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_A_shared.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B_shared.weight)
        self.lora_dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, sequence_length, hidden_dim = x.shape
        original_output = self.original_layer(x)
        x_flat = x.reshape(-1, hidden_dim)

        router_logits = self.router(x_flat)
        routing_weights_before = F.softmax(router_logits, dim=1, dtype=torch.float)
        routing_weights, selected_experts = torch.topk(routing_weights_before, self.top_k, dim=-1)
        routing_weights /= routing_weights.sum(dim=-1, keepdim=True)
        routing_weights = routing_weights.to(x_flat.dtype)
        expert_mask = F.one_hot(selected_experts, num_classes=self.num_experts).permute(2, 1, 0)

        x_lora = self.lora_A_shared(self.lora_dropout(x_flat))
        combined_expert_output = torch.zeros(
            (batch_size * sequence_length, self.shared_rank), dtype=x_lora.dtype, device=x_lora.device
        )

        for expert_idx in range(self.num_experts):
            idx, top_x = torch.where(expert_mask[expert_idx])
            if top_x.shape[0] == 0:
                continue
            expert_input = x_lora[top_x]
            expert_output = self.lora_A_experts[str(expert_idx)](self.lora_B_experts[str(expert_idx)](expert_input))
            current_expert_output = expert_output * routing_weights[top_x, idx, None]
            combined_expert_output.index_add_(0, top_x, current_expert_output.to(x_lora.dtype))

        final_lora_output = self.lora_B_shared(combined_expert_output)
        final_lora_output = final_lora_output.reshape(batch_size, sequence_length, -1)
        final_lora_output = final_lora_output * self.scaling

        if self.training:
            P = routing_weights_before
            imp = P.mean(dim=0)
            with torch.no_grad():
                assign_counts = torch.bincount(
                    selected_experts.reshape(-1), minlength=self.num_experts
                ).float()
                load = assign_counts / assign_counts.sum().clamp_min(1.0)
            self._lb_loss = self.num_experts * (imp * load).sum()
        else:
            self._lb_loss = None

        return original_output + final_lora_output

In [ ]:
class LBTrainer(Trainer):
    """ Custom Trainer to incorporate the load balancing loss. """
    def __init__(self, *args, lb_coef: float = 1e-2, **kwargs):
        super().__init__(*args, **kwargs)
        self.lb_coef = lb_coef

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        loss, outputs = super().compute_loss(model, inputs, return_outputs=True, **kwargs)
        aux_loss = None
        for m in model.modules():
            if hasattr(m, "_lb_loss"):
                lb = getattr(m, "_lb_loss", None)
                if lb is not None:
                    aux_loss = lb if aux_loss is None else (aux_loss + lb)
        if aux_loss is not None and self.is_in_train:
            loss = loss + self.lb_coef * aux_loss
        return (loss, outputs) if return_outputs else loss

In [ ]:
def patch_roberta_with_edge_aware_lora(model, **kwargs):
    """ Replaces target linear layers in RoBERTa with our custom MoE LoRA layer. """
    for layer in model.encoder.layer:
        layer.attention.self.query = EdgeAwareLoRALinear(layer.attention.self.query, **kwargs)
        layer.attention.self.value = EdgeAwareLoRALinear(layer.attention.self.value, **kwargs)
    return model

def compute_metrics(eval_pred):
    """ Calculates evaluation metrics (accuracy, F1, etc.). """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

In [ ]:
print("Loading and preparing MRPC dataset...")
dataset = load_dataset("glue", "mrpc")
tokenizer = RobertaTokenizer.from_pretrained("roberta-large")

def tokenize(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True, padding="max_length", max_length=128)

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

train_dataset = dataset["train"]
eval_dataset = dataset["test"]
print("Dataset preparation finished.")

Loading and preparing MRPC dataset...


Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

Dataset preparation finished.


In [ ]:
seeds = [42, 123, 7, 99, 101]
lb_coef = 1e-2
all_results = []

In [ ]:
for seed in seeds:
    print(f"\n{'='*40}\n  STARTING RUN: SEED={seed}, LB_COEF={lb_coef}\n{'='*40}\n")

    set_seed(seed)
    out_dir = f"./ourlora_mrpc_results/seed{seed}"
    os.makedirs(out_dir, exist_ok=True)

    model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=2)
    model.roberta = patch_roberta_with_edge_aware_lora(
        model.roberta, num_experts=4, top_k=2, shared_rank=4, expert_hidden_dim=32
    )

    trainable_modules = ["router", "lora_A_shared", "lora_B_shared", "lora_B_experts", "lora_A_experts"]
    for name, param in model.named_parameters():
        if not any(trainable_module in name for trainable_module in trainable_modules):
            param.requires_grad = False

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable Params: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")

    training_args = TrainingArguments(
        output_dir=f"./run_mrpc_results/seed{seed}",
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=3e-4,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        num_train_epochs=20,
        weight_decay=0.01,
        warmup_ratio=0.06,
        lr_scheduler_type="linear",
        logging_dir=f"./logs_mrpc/seed{seed}",
        logging_steps=20,
        report_to="none",
        fp16=True,
        seed=seed,
    )

    trainer = LBTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        lb_coef=lb_coef,
    )

    print(f"Starting training...")
    trainer.train()
    print(f"Training finished.")

    print(f"\n--- Analysis and Saving for SEED={seed} ---")

    log_history_df = pd.DataFrame(trainer.state.log_history)
    eval_logs = log_history_df.dropna(subset=['eval_loss'])

    if not eval_logs.empty:
        final_eval_metrics = eval_logs.iloc[-1].to_dict()
        final_eval_metrics['seed'] = seed
        all_results.append(final_eval_metrics)
    else:
        print("No evaluation logs found for this run.")

    print(f"Results for this run saved to {out_dir}")


  STARTING RUN: SEED=42, LB_COEF=0.01



Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable Params: 638,976 / 356,000,770 (0.18%)
Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.176100,0.635894,0.664928,0.442129,0.664928,0.531109
2,1.072100,0.548386,0.720580,0.728016,0.720580,0.723539
3,0.963700,0.500256,0.790725,0.785600,0.790725,0.784019
4,0.945000,0.415162,0.822609,0.821578,0.822609,0.815227
5,0.853900,0.380169,0.849275,0.847165,0.849275,0.846904
6,0.812900,0.322534,0.866087,0.865036,0.866087,0.865392
7,0.784700,0.322108,0.870145,0.869604,0.870145,0.867114
8,0.790900,0.302375,0.878841,0.882615,0.878841,0.879982
9,0.723500,0.329879,0.873043,0.873838,0.873043,0.873388
10,0.681500,0.348895,0.879420,0.879287,0.879420,0.876680


/home/luyuw1/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


...

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable Params: 638,976 / 356,000,770 (0.18%)
Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.122900,0.640652,0.664928,0.442129,0.664928,0.531109
2,1.072600,0.549093,0.689855,0.712703,0.689855,0.600518
3,0.964700,0.429462,0.816812,0.813316,0.816812,0.812310
4,0.861300,0.413883,0.845217,0.846380,0.845217,0.838948
5,0.863600,0.378444,0.844638,0.845278,0.844638,0.838628
6,0.811000,0.435323,0.848116,0.854745,0.848116,0.839636
7,0.764700,0.351977,0.859130,0.859675,0.859130,0.854521
8,0.729700,0.351965,0.864928,0.863818,0.864928,0.862062
9,0.717800,0.336460,0.868986,0.867514,0.868986,0.867559
10,0.699500,0.329385,0.878261,0.877370,0.878261,0.877658


/home/luyuw1/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Training finished.

--- Analysis and Saving for SEED=101 ---
Results for this run saved to ./ourlora_mrpc_results/seed101


In [ ]:
if all_results:
    results_df = pd.DataFrame(all_results)
    print("--- Final Evaluation Results per Seed ---")
    print(results_df[['seed', 'eval_accuracy', 'eval_loss']].to_string(float_format="%.4f"))

    # --- CHANGE 3: Calculate and print average, best, and std dev ---
    avg_accuracy = results_df['eval_accuracy'].mean()
    best_accuracy = results_df['eval_accuracy'].max()
    std_dev_accuracy = results_df['eval_accuracy'].std()

    print("\n--- Summary Across 5 Seeds on MRPC ---")
    print(f"📊 Average Accuracy: {avg_accuracy:.4f}")
    print(f"🏆 Best Accuracy:    {best_accuracy:.4f}")
    print(f"📈 Std Deviation:    {std_dev_accuracy:.4f}")
else:
    print("No results to summarize.")

print("\nExperiment finished successfully! 🚀")

--- Final Evaluation Results per Seed ---
   seed  eval_accuracy  eval_loss
0    42         0.8603     0.4955
1   123         0.8759     0.4789
2     7         0.8707     0.5065
3    99         0.8754     0.4991
4   101         0.8707     0.4870

--- Summary Across 5 Seeds on MRPC ---
📊 Average Accuracy: 0.8706
🏆 Best Accuracy:    0.8759
📈 Std Deviation:    0.0063

Experiment finished successfully! 🚀


# OurLoRA with aux for CoLA roberta-large

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, matthews_corrcoef
import os

# Enable memory-efficient Scaled Dot-Product Attention (SDPA) if using PyTorch 2.0+
if torch.cuda.is_available():
    try:
        import packaging.version as V
        if V.Version(torch.__version__) >= V.Version("2.0"):
            torch.backends.cuda.enable_mem_efficient_sdp(True)
            print("Enabled Scaled Dot-Product Attention (SDPA).")
    except Exception:
        pass

Enabled Scaled Dot-Product Attention (SDPA).


In [ ]:
class EdgeAwareLoRALinear(nn.Module):
    """
    Custom PyTorch module implementing a Mixture of Experts (MoE) LoRA.
    It replaces a standard linear layer, using a router to select expert pathways.
    """
    def __init__(self, original_layer, num_experts=4, top_k=2, shared_rank=4, expert_hidden_dim=32, lora_alpha=4, dropout=0.1):
        super().__init__()

        self.num_experts = num_experts
        self.top_k = top_k
        self.shared_rank = shared_rank
        self.expert_hidden_dim = expert_hidden_dim
        self.lora_alpha = lora_alpha
        self.scaling = self.lora_alpha / self.shared_rank
        self.original_layer = original_layer
        self._lb_loss = None # Initialize lb_loss attribute

        in_features = original_layer.in_features
        out_features = original_layer.out_features

        self.router = nn.Linear(in_features, self.num_experts, bias=False)
        self.lora_A_shared = nn.Linear(in_features, self.shared_rank, bias=False)

        self.lora_B_experts = nn.ModuleDict()
        self.lora_A_experts = nn.ModuleDict()

        for i in range(self.num_experts):
            expert_name = str(i)
            self.lora_B_experts[expert_name] = nn.Linear(self.shared_rank, self.expert_hidden_dim, bias=False)
            self.lora_A_experts[expert_name] = nn.Linear(self.expert_hidden_dim, self.shared_rank, bias=False)
            nn.init.kaiming_uniform_(self.lora_B_experts[expert_name].weight, a=math.sqrt(5))
            nn.init.kaiming_uniform_(self.lora_A_experts[expert_name].weight, a=math.sqrt(5))

        self.lora_B_shared = nn.Linear(self.shared_rank, out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_A_shared.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B_shared.weight)
        self.lora_dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, sequence_length, hidden_dim = x.shape
        original_output = self.original_layer(x)
        x_flat = x.reshape(-1, hidden_dim)

        router_logits = self.router(x_flat)
        routing_weights_before = F.softmax(router_logits, dim=1, dtype=torch.float)
        routing_weights, selected_experts = torch.topk(routing_weights_before, self.top_k, dim=-1)
        routing_weights /= routing_weights.sum(dim=-1, keepdim=True)
        routing_weights = routing_weights.to(x_flat.dtype)
        expert_mask = F.one_hot(selected_experts, num_classes=self.num_experts).permute(2, 1, 0)

        x_lora = self.lora_A_shared(self.lora_dropout(x_flat))
        combined_expert_output = torch.zeros(
            (batch_size * sequence_length, self.shared_rank), dtype=x_lora.dtype, device=x_lora.device
        )

        for expert_idx in range(self.num_experts):
            idx, top_x = torch.where(expert_mask[expert_idx])
            if top_x.shape[0] == 0:
                continue
            expert_input = x_lora[top_x]
            expert_output = self.lora_A_experts[str(expert_idx)](self.lora_B_experts[str(expert_idx)](expert_input))
            current_expert_output = expert_output * routing_weights[top_x, idx, None]
            combined_expert_output.index_add_(0, top_x, current_expert_output.to(x_lora.dtype))

        final_lora_output = self.lora_B_shared(combined_expert_output)
        final_lora_output = final_lora_output.reshape(batch_size, sequence_length, -1)
        final_lora_output = final_lora_output * self.scaling

        if self.training:
            P = routing_weights_before
            imp = P.mean(dim=0)
            with torch.no_grad():
                assign_counts = torch.bincount(
                    selected_experts.reshape(-1), minlength=self.num_experts
                ).float()
                load = assign_counts / assign_counts.sum().clamp_min(1.0)
            self._lb_loss = self.num_experts * (imp * load).sum()
        else:
            self._lb_loss = None

        return original_output + final_lora_output

In [ ]:
class LBTrainer(Trainer):
    """ Custom Trainer to incorporate the load balancing loss. """
    def __init__(self, *args, lb_coef: float = 1e-2, **kwargs):
        super().__init__(*args, **kwargs)
        self.lb_coef = lb_coef

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        loss, outputs = super().compute_loss(model, inputs, return_outputs=True, **kwargs)
        aux_loss = None
        for m in model.modules():
            if hasattr(m, "_lb_loss"):
                lb = getattr(m, "_lb_loss", None)
                if lb is not None:
                    aux_loss = lb if aux_loss is None else (aux_loss + lb)
        if aux_loss is not None and self.is_in_train:
            loss = loss + self.lb_coef * aux_loss
        return (loss, outputs) if return_outputs else loss

In [ ]:
def patch_roberta_with_edge_aware_lora(model, **kwargs):
    """ Replaces target linear layers in RoBERTa with our custom MoE LoRA layer. """
    for layer in model.encoder.layer:
        layer.attention.self.query = EdgeAwareLoRALinear(layer.attention.self.query, **kwargs)
        layer.attention.self.value = EdgeAwareLoRALinear(layer.attention.self.value, **kwargs)
    return model

# --- Add Matthews Correlation Coefficient (MCC) for CoLA ---
def compute_metrics(eval_pred):
    """ Calculates evaluation metrics, including MCC for CoLA. """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    mcc = matthews_corrcoef(labels, preds)
    return {"accuracy": acc, "mcc": mcc}

In [ ]:
print("Loading and preparing CoLA dataset...")
dataset = load_dataset("glue", "cola")
tokenizer = RobertaTokenizer.from_pretrained("roberta-large")

def tokenize(example):
    return tokenizer(example["sentence"], truncation=True, padding="max_length", max_length=128)

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# CoLA has official train/validation splits
train_dataset = dataset["train"]
eval_dataset = dataset["validation"]
print("Dataset preparation finished.")

Loading and preparing CoLA dataset...
Dataset preparation finished.


In [ ]:
seeds = [42, 123, 7, 99, 101]
lb_coef = 1e-2
all_results = []

In [ ]:
for seed in seeds:
    print(f"\n{'='*40}\n  STARTING RUN: SEED={seed}, LB_COEF={lb_coef}\n{'='*40}\n")

    set_seed(seed)
    # --- CHANGE 4: Update output directories for CoLA ---
    out_dir = f"./ourlora_cola_results/seed{seed}"
    os.makedirs(out_dir, exist_ok=True)

    # num_labels=2 is correct for CoLA (acceptable or not)
    model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=2)
    model.roberta = patch_roberta_with_edge_aware_lora(
        model.roberta, num_experts=4, top_k=2, shared_rank=4, lora_alpha=4
    )

    trainable_modules = ["router", "lora_A_shared", "lora_B_shared", "lora_B_experts", "lora_A_experts"]
    for name, param in model.named_parameters():
        if not any(trainable_module in name for trainable_module in trainable_modules):
            param.requires_grad = False

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable Params: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")

    training_args = TrainingArguments(
        output_dir=f"./run_cola_results/seed{seed}",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="mcc",
        learning_rate=2e-4,
        per_device_train_batch_size=64,
        per_device_eval_batch_size=64,
        num_train_epochs=20,
        weight_decay=0.01,
        warmup_ratio=0.06,
        lr_scheduler_type="linear",
        logging_dir=f"./logs_cola/seed{seed}",
        logging_steps=50,
        report_to="none",
        fp16=True,
        seed=seed,
    )

    trainer = LBTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        lb_coef=lb_coef,
    )

    print(f"Starting training...")
    trainer.train()
    print(f"Training finished.")

    print(f"\n--- Analysis and Saving for SEED={seed} ---")

    # Evaluate the best model on the validation set
    final_metrics = trainer.evaluate()
    final_metrics['seed'] = seed
    all_results.append(final_metrics)


  STARTING RUN: SEED=42, LB_COEF=0.01



Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable Params: 638,976 / 356,000,770 (0.18%)
Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Mcc
1,1.269800,0.621119,0.671141,0.048462
2,1.003500,0.483895,0.783317,0.452809
3,0.913900,0.447129,0.827421,0.577653
4,0.893700,0.474780,0.833174,0.590675
5,0.851500,0.436761,0.838926,0.605661
6,0.835800,0.416129,0.842761,0.615717
7,0.826200,0.454602,0.834132,0.593539
8,0.818600,0.397830,0.849473,0.633321
9,0.781400,0.373589,0.860019,0.660535
10,0.772500,0.449613,0.841802,0.614278


Training finished.

--- Analysis and Saving for SEED=42 ---


...


  STARTING RUN: SEED=101, LB_COEF=0.01



Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable Params: 638,976 / 356,000,770 (0.18%)
Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Mcc
1,1.132600,0.615658,0.691275,0.000000
2,1.059000,0.526894,0.691275,0.000000
3,0.923500,0.501346,0.803452,0.511561
4,0.870300,0.454996,0.825503,0.570552
5,0.854000,0.389591,0.840844,0.611508
6,0.822600,0.431866,0.843720,0.618194
7,0.824600,0.394262,0.850431,0.635993
8,0.800700,0.386813,0.854267,0.645483
9,0.790200,0.394317,0.856184,0.650509
10,0.787000,0.373519,0.857143,0.652922


Training finished.

--- Analysis and Saving for SEED=101 ---


In [ ]:
if all_results:
    results_df = pd.DataFrame(all_results)
    print("--- Final Evaluation Results per Seed ---")
    # --- CHANGE 6: Display eval_mcc in the summary table ---
    print(results_df[['seed', 'eval_accuracy', 'eval_mcc', 'eval_loss']].to_string(float_format="%.4f"))

    avg_mcc = results_df['eval_mcc'].mean()
    best_mcc = results_df['eval_mcc'].max()
    std_dev_mcc = results_df['eval_mcc'].std()

    avg_accuracy = results_df['eval_accuracy'].mean()
    best_accuracy = results_df['eval_accuracy'].max()

    print("\n--- Summary Across 5 Seeds on CoLA ---")
    print(f"🎯 Average MCC: {avg_mcc:.4f}")
    print(f"🏆 Best MCC:    {best_mcc:.4f}")
    print(f"📈 Std Dev MCC: {std_dev_mcc:.4f}")
    print("---")
    print(f"📊 Average Accuracy: {avg_accuracy:.4f}")
    print(f"✨ Best Accuracy:    {best_accuracy:.4f}")
else:
    print("No results to summarize.")

print("\nExperiment finished successfully! 🚀")

--- Final Evaluation Results per Seed ---
   seed  eval_accuracy  eval_mcc  eval_loss
0    42         0.8600    0.6605     0.3736
1   123         0.8543    0.6456     0.3752
2     7         0.8543    0.6455     0.4252
3    99         0.8562    0.6509     0.4137
4   101         0.8571    0.6529     0.3735

--- Summary Across 5 Seeds on CoLA ---
🎯 Average MCC: 0.6511
🏆 Best MCC:    0.6605
📈 Std Dev MCC: 0.0062
---
📊 Average Accuracy: 0.8564
✨ Best Accuracy:    0.8600

Experiment finished successfully! 🚀


# OurLoRA with aux for STS-B roberta-large

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)
from datasets import load_dataset
from scipy.stats import pearsonr, spearmanr
import os

if torch.cuda.is_available():
    try:
        import packaging.version as V
        if V.Version(torch.__version__) >= V.Version("2.0"):
            torch.backends.cuda.enable_mem_efficient_sdp(True)
            print("Enabled Scaled Dot-Product Attention (SDPA).")
    except Exception:
        pass

Enabled Scaled Dot-Product Attention (SDPA).


In [ ]:
class EdgeAwareLoRALinear(nn.Module):
    """
    Custom PyTorch module implementing a Mixture of Experts (MoE) LoRA.
    It replaces a standard linear layer, using a router to select expert pathways.
    """
    def __init__(self, original_layer, num_experts=4, top_k=2, shared_rank=4, expert_hidden_dim=32, lora_alpha=4, dropout=0.1):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.shared_rank = shared_rank
        self.expert_hidden_dim = expert_hidden_dim
        self.lora_alpha = lora_alpha
        self.scaling = self.lora_alpha / self.shared_rank
        self.original_layer = original_layer
        self._lb_loss = None

        in_features = original_layer.in_features
        out_features = original_layer.out_features

        self.router = nn.Linear(in_features, self.num_experts, bias=False)
        self.lora_A_shared = nn.Linear(in_features, self.shared_rank, bias=False)
        self.lora_B_experts = nn.ModuleDict()
        self.lora_A_experts = nn.ModuleDict()
        for i in range(self.num_experts):
            expert_name = str(i)
            self.lora_B_experts[expert_name] = nn.Linear(self.shared_rank, self.expert_hidden_dim, bias=False)
            self.lora_A_experts[expert_name] = nn.Linear(self.expert_hidden_dim, self.shared_rank, bias=False)
            nn.init.kaiming_uniform_(self.lora_B_experts[expert_name].weight, a=math.sqrt(5))
            nn.init.kaiming_uniform_(self.lora_A_experts[expert_name].weight, a=math.sqrt(5))
        self.lora_B_shared = nn.Linear(self.shared_rank, out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_A_shared.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B_shared.weight)
        self.lora_dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, sequence_length, hidden_dim = x.shape
        original_output = self.original_layer(x)
        x_flat = x.reshape(-1, hidden_dim)
        router_logits = self.router(x_flat)
        routing_weights_before = F.softmax(router_logits, dim=1, dtype=torch.float)
        routing_weights, selected_experts = torch.topk(routing_weights_before, self.top_k, dim=-1)
        routing_weights /= routing_weights.sum(dim=-1, keepdim=True)
        routing_weights = routing_weights.to(x_flat.dtype)
        expert_mask = F.one_hot(selected_experts, num_classes=self.num_experts).permute(2, 1, 0)
        x_lora = self.lora_A_shared(self.lora_dropout(x_flat))
        combined_expert_output = torch.zeros((batch_size * sequence_length, self.shared_rank), dtype=x_lora.dtype, device=x_lora.device)
        for expert_idx in range(self.num_experts):
            idx, top_x = torch.where(expert_mask[expert_idx])
            if top_x.shape[0] == 0:
                continue
            expert_input = x_lora[top_x]
            expert_output = self.lora_A_experts[str(expert_idx)](self.lora_B_experts[str(expert_idx)](expert_input))
            current_expert_output = expert_output * routing_weights[top_x, idx, None]
            combined_expert_output.index_add_(0, top_x, current_expert_output.to(x_lora.dtype))
        final_lora_output = self.lora_B_shared(combined_expert_output)
        final_lora_output = final_lora_output.reshape(batch_size, sequence_length, -1)
        final_lora_output = final_lora_output * self.scaling
        if self.training:
            P = routing_weights_before
            imp = P.mean(dim=0)
            with torch.no_grad():
                assign_counts = torch.bincount(selected_experts.reshape(-1), minlength=self.num_experts).float()
                load = assign_counts / assign_counts.sum().clamp_min(1.0)
            self._lb_loss = self.num_experts * (imp * load).sum()
        else:
            self._lb_loss = None
        return original_output + final_lora_output


In [ ]:
class LBTrainer(Trainer):
    def __init__(self, *args, lb_coef: float = 1e-2, **kwargs):
        super().__init__(*args, **kwargs)
        self.lb_coef = lb_coef
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        loss, outputs = super().compute_loss(model, inputs, return_outputs=True, **kwargs)
        aux_loss = None
        for m in model.modules():
            if hasattr(m, "_lb_loss"):
                lb = getattr(m, "_lb_loss", None)
                if lb is not None:
                    aux_loss = lb if aux_loss is None else (aux_loss + lb)
        if aux_loss is not None and self.is_in_train:
            loss = loss + self.lb_coef * aux_loss
        return (loss, outputs) if return_outputs else loss

In [ ]:
def patch_roberta_with_edge_aware_lora(model, **kwargs):
    for layer in model.encoder.layer:
        layer.attention.self.query = EdgeAwareLoRALinear(layer.attention.self.query, **kwargs)
        layer.attention.self.value = EdgeAwareLoRALinear(layer.attention.self.value, **kwargs)
    return model

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.squeeze(logits) # Squeeze predictions for regression
    pearson_corr, _ = pearsonr(preds, labels)
    spearman_corr, _ = spearmanr(preds, labels)
    return {
        "pearson": pearson_corr,
        "spearmanr": spearman_corr,
    }

In [ ]:
print("Loading and preparing STS-B dataset...")
dataset = load_dataset("glue", "stsb")
tokenizer = RobertaTokenizer.from_pretrained("roberta-large")

def tokenize(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True, padding="max_length", max_length=512)

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

train_dataset = dataset["train"]
eval_dataset = dataset["validation"]
print("Dataset preparation finished.")

Loading and preparing STS-B dataset...
Dataset preparation finished.


In [ ]:
seeds = [42, 123, 7, 99, 101]
lb_coef = 1e-2
all_results = []

In [ ]:
for seed in seeds:
    print(f"\n{'='*40}\n  STARTING RUN: SEED={seed}, LB_COEF={lb_coef}\n{'='*40}\n")
    set_seed(seed)
    out_dir = f"./ourlora_stsb_results/seed{seed}"
    os.makedirs(out_dir, exist_ok=True)

    model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=1)
    model.roberta = patch_roberta_with_edge_aware_lora(model.roberta, num_experts=4, top_k=2, shared_rank=4, lora_alpha=4)

    trainable_modules = ["router", "lora_A_shared", "lora_B_shared", "lora_B_experts", "lora_A_experts"]
    for name, param in model.named_parameters():
        if not any(trainable_module in name for trainable_module in trainable_modules):
            param.requires_grad = False
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable Params: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")

    training_args = TrainingArguments(
        output_dir=f"./run_stsb_results/seed{seed}",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="pearson",
        learning_rate=2e-4,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=64,
        num_train_epochs=30,
        weight_decay=0.01,
        warmup_ratio=0.06,
        lr_scheduler_type="linear",
        logging_dir=f"./logs_stsb/seed{seed}",
        logging_steps=50,
        report_to="none",
        fp16=True,
        seed=seed,
    )

    trainer = LBTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        lb_coef=lb_coef,
    )

    print(f"Starting training...")
    trainer.train()
    print(f"Training finished.")
    print(f"\n--- Analysis and Saving for SEED={seed} ---")

    final_metrics = trainer.evaluate()
    final_metrics['seed'] = seed
    all_results.append(final_metrics)



  STARTING RUN: SEED=42, LB_COEF=0.01



Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable Params: 638,976 / 355,999,745 (0.18%)
Starting training...


Epoch,Training Loss,Validation Loss,Pearson,Spearmanr
1,6.251800,3.523135,-0.020305,-0.003770
2,2.566400,1.984124,0.663833,0.666430
3,1.155300,0.536582,0.877358,0.881495
4,0.987600,0.499106,0.890911,0.897955
5,0.915600,0.417967,0.908767,0.905655
6,0.902300,0.428248,0.907283,0.907157
7,0.853400,0.392204,0.913457,0.909854
8,0.817200,0.426941,0.910944,0.910387
9,0.813500,0.401618,0.915441,0.911681
10,0.794200,0.401600,0.914922,0.913396


Training finished.

--- Analysis and Saving for SEED=42 ---


...


  STARTING RUN: SEED=101, LB_COEF=0.01



Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable Params: 638,976 / 355,999,745 (0.18%)
Starting training...


Epoch,Training Loss,Validation Loss,Pearson,Spearmanr
1,8.489800,3.464445,-0.016519,0.014883
2,2.867800,2.326230,0.381210,0.387008
3,1.448400,0.727951,0.832509,0.856665
4,1.082200,0.584070,0.884972,0.893697
5,0.965800,0.542191,0.896820,0.904542
6,0.882500,0.431603,0.909771,0.910701
7,0.899900,0.413647,0.909370,0.912408
8,0.848500,0.447814,0.913208,0.912539
9,0.816700,0.390978,0.916737,0.916278
10,0.810300,0.419878,0.913377,0.915631


Training finished.

--- Analysis and Saving for SEED=101 ---


In [ ]:
if all_results:
    results_df = pd.DataFrame(all_results)
    print("--- Final Evaluation Results per Seed ---")
    # --- CHANGE 7: Display regression metrics in the summary table ---
    print(results_df[['seed', 'eval_pearson', 'eval_spearmanr', 'eval_loss']].to_string(float_format="%.4f"))

    avg_pearson = results_df['eval_pearson'].mean()
    best_pearson = results_df['eval_pearson'].max()
    std_dev_pearson = results_df['eval_pearson'].std()

    print("\n--- Summary Across 5 Seeds on STS-B ---")
    print(f"🎯 Average Pearson: {avg_pearson:.4f}")
    print(f"🏆 Best Pearson:    {best_pearson:.4f}")
    print(f"📈 Std Dev Pearson: {std_dev_pearson:.4f}")
else:
    print("No results to summarize.")

print("\nExperiment finished successfully! 🚀")

--- Final Evaluation Results per Seed ---
   seed  eval_pearson  eval_spearmanr  eval_loss
0    42        0.9187          0.9153     0.3802
1   123        0.9167          0.9144     0.4086
2     7        0.9201          0.9152     0.3980
3    99        0.9193          0.9163     0.3842
4   101        0.9197          0.9175     0.3836

--- Summary Across 5 Seeds on STS-B ---
🎯 Average Pearson: 0.9189
🏆 Best Pearson:    0.9201
📈 Std Dev Pearson: 0.0013

Experiment finished successfully! 🚀


# OurLoRA with aux for RTE roberta-large

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score
import os

# Enable memory-efficient Scaled Dot-Product Attention (SDPA) if using PyTorch 2.0+
if torch.cuda.is_available():
    try:
        import packaging.version as V
        if V.Version(torch.__version__) >= V.Version("2.0"):
            torch.backends.cuda.enable_mem_efficient_sdp(True)
            print("Enabled Scaled Dot-Product Attention (SDPA).")
    except Exception:
        pass

Enabled Scaled Dot-Product Attention (SDPA).


In [ ]:
class EdgeAwareLoRALinear(nn.Module):
    """
    Custom PyTorch module implementing a Mixture of Experts (MoE) LoRA.
    It replaces a standard linear layer, using a router to select expert pathways.
    """
    def __init__(self, original_layer, num_experts=4, top_k=2, shared_rank=4, expert_hidden_dim=32, lora_alpha=4, dropout=0.1):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.shared_rank = shared_rank
        self.expert_hidden_dim = expert_hidden_dim
        self.lora_alpha = lora_alpha
        self.scaling = self.lora_alpha / self.shared_rank
        self.original_layer = original_layer
        self._lb_loss = None

        in_features = original_layer.in_features
        out_features = original_layer.out_features

        self.router = nn.Linear(in_features, self.num_experts, bias=False)
        self.lora_A_shared = nn.Linear(in_features, self.shared_rank, bias=False)
        self.lora_B_experts = nn.ModuleDict()
        self.lora_A_experts = nn.ModuleDict()
        for i in range(self.num_experts):
            expert_name = str(i)
            self.lora_B_experts[expert_name] = nn.Linear(self.shared_rank, self.expert_hidden_dim, bias=False)
            self.lora_A_experts[expert_name] = nn.Linear(self.expert_hidden_dim, self.shared_rank, bias=False)
            nn.init.kaiming_uniform_(self.lora_B_experts[expert_name].weight, a=math.sqrt(5))
            nn.init.kaiming_uniform_(self.lora_A_experts[expert_name].weight, a=math.sqrt(5))
        self.lora_B_shared = nn.Linear(self.shared_rank, out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_A_shared.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B_shared.weight)
        self.lora_dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, sequence_length, hidden_dim = x.shape
        original_output = self.original_layer(x)
        x_flat = x.reshape(-1, hidden_dim)
        router_logits = self.router(x_flat)
        routing_weights_before = F.softmax(router_logits, dim=1, dtype=torch.float)
        routing_weights, selected_experts = torch.topk(routing_weights_before, self.top_k, dim=-1)
        routing_weights /= routing_weights.sum(dim=-1, keepdim=True)
        routing_weights = routing_weights.to(x_flat.dtype)
        expert_mask = F.one_hot(selected_experts, num_classes=self.num_experts).permute(2, 1, 0)
        x_lora = self.lora_A_shared(self.lora_dropout(x_flat))
        combined_expert_output = torch.zeros((batch_size * sequence_length, self.shared_rank), dtype=x_lora.dtype, device=x_lora.device)
        for expert_idx in range(self.num_experts):
            idx, top_x = torch.where(expert_mask[expert_idx])
            if top_x.shape[0] == 0:
                continue
            expert_input = x_lora[top_x]
            expert_output = self.lora_A_experts[str(expert_idx)](self.lora_B_experts[str(expert_idx)](expert_input))
            current_expert_output = expert_output * routing_weights[top_x, idx, None]
            combined_expert_output.index_add_(0, top_x, current_expert_output.to(x_lora.dtype))
        final_lora_output = self.lora_B_shared(combined_expert_output)
        final_lora_output = final_lora_output.reshape(batch_size, sequence_length, -1)
        final_lora_output = final_lora_output * self.scaling
        if self.training:
            P = routing_weights_before
            imp = P.mean(dim=0)
            with torch.no_grad():
                assign_counts = torch.bincount(selected_experts.reshape(-1), minlength=self.num_experts).float()
                load = assign_counts / assign_counts.sum().clamp_min(1.0)
            self._lb_loss = self.num_experts * (imp * load).sum()
        else:
            self._lb_loss = None
        return original_output + final_lora_output

In [ ]:
class LBTrainer(Trainer):
    def __init__(self, *args, lb_coef: float = 1e-2, **kwargs):
        super().__init__(*args, **kwargs)
        self.lb_coef = lb_coef
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        loss, outputs = super().compute_loss(model, inputs, return_outputs=True, **kwargs)
        aux_loss = None
        for m in model.modules():
            if hasattr(m, "_lb_loss"):
                lb = getattr(m, "_lb_loss", None)
                if lb is not None:
                    aux_loss = lb if aux_loss is None else (aux_loss + lb)
        if aux_loss is not None and self.is_in_train:
            loss = loss + self.lb_coef * aux_loss
        return (loss, outputs) if return_outputs else loss

In [ ]:
def patch_roberta_with_edge_aware_lora(model, **kwargs):
    for layer in model.encoder.layer:
        layer.attention.self.query = EdgeAwareLoRALinear(layer.attention.self.query, **kwargs)
        layer.attention.self.value = EdgeAwareLoRALinear(layer.attention.self.value, **kwargs)
    return model

# --- CHANGE 1: Update metrics function for classification (Accuracy) ---
def compute_metrics(eval_pred):
    """ Calculates evaluation metrics for classification. """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

In [ ]:
print("Loading and preparing RTE dataset...")
dataset = load_dataset("glue", "rte")
tokenizer = RobertaTokenizer.from_pretrained("roberta-large")

# RTE uses two sentences
def tokenize(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True, padding="max_length", max_length=512) # Increased max_length for RTE

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

train_dataset = dataset["train"]
eval_dataset = dataset["validation"]
print("Dataset preparation finished.")

Loading and preparing RTE dataset...


Map:   0%|          | 0/2490 [00:00<?, ? examples/s]

Map:   0%|          | 0/277 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Dataset preparation finished.


In [ ]:
seeds = [42, 123, 7, 99, 101]
lb_coef = 1e-2
all_results = []

In [ ]:
for seed in seeds:
    print(f"\n{'='*40}\n  STARTING RUN: SEED={seed}, LB_COEF={lb_coef}\n{'='*40}\n")
    set_seed(seed)
    # --- CHANGE 3: Update output directories for RTE ---
    out_dir = f"./ourlora_rte_results/seed{seed}"
    os.makedirs(out_dir, exist_ok=True)

    # --- CHANGE 4: Set num_labels=2 for binary classification ---
    model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=2)
    model.roberta = patch_roberta_with_edge_aware_lora(model.roberta, num_experts=4, top_k=2, shared_rank=4, lora_alpha=4)

    trainable_modules = ["router", "lora_A_shared", "lora_B_shared", "lora_B_experts", "lora_A_experts"]
    for name, param in model.named_parameters():
        if not any(trainable_module in name for trainable_module in trainable_modules):
            param.requires_grad = False
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable Params: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")

    # --- CHANGE 5: Adjust training arguments for a very small dataset ---
    training_args = TrainingArguments(
        output_dir=f"./run_rte_results/seed{seed}",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        learning_rate=4e-4, # Lower learning rate for stability
        per_device_train_batch_size=32, # Small batch size
        per_device_eval_batch_size=64,
        num_train_epochs=20, # Train for more epochs, but load the best model
        weight_decay=0.01,
        warmup_ratio=0.1, # Larger warmup for stability
        lr_scheduler_type="linear",
        logging_dir=f"./logs_rte/seed{seed}",
        logging_steps=20,
        report_to="none",
        fp16=True,
        seed=seed,
    )

    trainer = LBTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        lb_coef=lb_coef,
    )

    print(f"Starting training...")
    trainer.train()
    print(f"Training finished.")
    print(f"\n--- Analysis and Saving for SEED={seed} ---")

    final_metrics = trainer.evaluate()
    final_metrics['seed'] = seed
    all_results.append(final_metrics)


  STARTING RUN: SEED=42, LB_COEF=0.01



Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable Params: 638,976 / 356,000,770 (0.18%)
Starting training...


Epoch,Training Loss,Validation Loss,Accuracy
1,1.209400,0.695662,0.458484
2,1.181900,0.690985,0.527076
3,1.182300,0.701992,0.527076
4,1.166200,0.667372,0.588448
5,1.111200,0.596088,0.685921
6,0.989100,0.505541,0.758123
7,0.907000,0.472854,0.805054
8,0.855000,0.502085,0.797834
9,0.738000,0.512605,0.812274
10,0.776900,0.418190,0.844765


Training finished.

--- Analysis and Saving for SEED=42 ---


...


  STARTING RUN: SEED=101, LB_COEF=0.01



Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable Params: 638,976 / 356,000,770 (0.18%)
Starting training...


Epoch,Training Loss,Validation Loss,Accuracy
1,1.184100,0.693596,0.490975
2,1.177900,0.695633,0.494585
3,1.176300,0.690610,0.563177
4,1.160800,0.662641,0.595668
5,1.050200,0.509026,0.765343
6,0.876400,0.480862,0.794224
7,0.786600,0.395545,0.815884
8,0.747600,0.451127,0.830325
9,0.716500,0.508246,0.823105
10,0.665100,0.736132,0.805054


Training finished.

--- Analysis and Saving for SEED=101 ---


In [ ]:
if all_results:
    results_df = pd.DataFrame(all_results)
    print("--- Final Evaluation Results per Seed ---")
    # --- CHANGE 6: Display accuracy in the summary table ---
    print(results_df[['seed', 'eval_accuracy', 'eval_loss']].to_string(float_format="%.4f"))

    avg_accuracy = results_df['eval_accuracy'].mean()
    best_accuracy = results_df['eval_accuracy'].max()
    std_dev_accuracy = results_df['eval_accuracy'].std()

    print("\n--- Summary Across 5 Seeds on RTE ---")
    print(f"📊 Average Accuracy: {avg_accuracy:.4f}")
    print(f"🏆 Best Accuracy:    {best_accuracy:.4f}")
    print(f"📈 Std Deviation:    {std_dev_accuracy:.4f}")
else:
    print("No results to summarize.")

print("\nExperiment finished successfully! 🚀")

--- Final Evaluation Results per Seed ---
   seed  eval_accuracy  eval_loss
0    42            NaN     0.3802
1   123            NaN     0.4086
2     7            NaN     0.3980
3    99            NaN     0.3842
4   101            NaN     0.3836
5    42         0.8448     0.4182
6   123         0.8412     0.6316
7     7         0.8375     0.4917
8    99         0.8556     0.5529
9   101         0.8448     0.7296

--- Summary Across 5 Seeds on RTE ---
📊 Average Accuracy: 0.8448
🏆 Best Accuracy:    0.8556
📈 Std Deviation:    0.0068

Experiment finished successfully! 🚀


# OurLoRA with aux for QNLI roberta-large

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score
import os

# Enable memory-efficient Scaled Dot-Product Attention (SDPA) if using PyTorch 2.0+
if torch.cuda.is_available():
    try:
        import packaging.version as V
        if V.Version(torch.__version__) >= V.Version("2.0"):
            torch.backends.cuda.enable_mem_efficient_sdp(True)
            print("Enabled Scaled Dot-Product Attention (SDPA).")
    except Exception:
        pass

Enabled Scaled Dot-Product Attention (SDPA).


In [ ]:
class EdgeAwareLoRALinear(nn.Module):
    """
    Custom PyTorch module implementing a Mixture of Experts (MoE) LoRA.
    It replaces a standard linear layer, using a router to select expert pathways.
    """
    def __init__(self, original_layer, num_experts=4, top_k=2, shared_rank=4, expert_hidden_dim=32, lora_alpha=4, dropout=0.1):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.shared_rank = shared_rank
        self.expert_hidden_dim = expert_hidden_dim
        self.lora_alpha = lora_alpha
        self.scaling = self.lora_alpha / self.shared_rank
        self.original_layer = original_layer
        self._lb_loss = None

        in_features = original_layer.in_features
        out_features = original_layer.out_features

        self.router = nn.Linear(in_features, self.num_experts, bias=False)
        self.lora_A_shared = nn.Linear(in_features, self.shared_rank, bias=False)
        self.lora_B_experts = nn.ModuleDict()
        self.lora_A_experts = nn.ModuleDict()
        for i in range(self.num_experts):
            expert_name = str(i)
            self.lora_B_experts[expert_name] = nn.Linear(self.shared_rank, self.expert_hidden_dim, bias=False)
            self.lora_A_experts[expert_name] = nn.Linear(self.expert_hidden_dim, self.shared_rank, bias=False)
            nn.init.kaiming_uniform_(self.lora_B_experts[expert_name].weight, a=math.sqrt(5))
            nn.init.kaiming_uniform_(self.lora_A_experts[expert_name].weight, a=math.sqrt(5))
        self.lora_B_shared = nn.Linear(self.shared_rank, out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_A_shared.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B_shared.weight)
        self.lora_dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, sequence_length, hidden_dim = x.shape
        original_output = self.original_layer(x)
        x_flat = x.reshape(-1, hidden_dim)
        router_logits = self.router(x_flat)
        routing_weights_before = F.softmax(router_logits, dim=1, dtype=torch.float)
        routing_weights, selected_experts = torch.topk(routing_weights_before, self.top_k, dim=-1)
        routing_weights /= routing_weights.sum(dim=-1, keepdim=True)
        routing_weights = routing_weights.to(x_flat.dtype)
        expert_mask = F.one_hot(selected_experts, num_classes=self.num_experts).permute(2, 1, 0)
        x_lora = self.lora_A_shared(self.lora_dropout(x_flat))
        combined_expert_output = torch.zeros((batch_size * sequence_length, self.shared_rank), dtype=x_lora.dtype, device=x_lora.device)
        for expert_idx in range(self.num_experts):
            idx, top_x = torch.where(expert_mask[expert_idx])
            if top_x.shape[0] == 0:
                continue
            expert_input = x_lora[top_x]
            expert_output = self.lora_A_experts[str(expert_idx)](self.lora_B_experts[str(expert_idx)](expert_input))
            current_expert_output = expert_output * routing_weights[top_x, idx, None]
            combined_expert_output.index_add_(0, top_x, current_expert_output.to(x_lora.dtype))
        final_lora_output = self.lora_B_shared(combined_expert_output)
        final_lora_output = final_lora_output.reshape(batch_size, sequence_length, -1)
        final_lora_output = final_lora_output * self.scaling
        if self.training:
            P = routing_weights_before
            imp = P.mean(dim=0)
            with torch.no_grad():
                assign_counts = torch.bincount(selected_experts.reshape(-1), minlength=self.num_experts).float()
                load = assign_counts / assign_counts.sum().clamp_min(1.0)
            self._lb_loss = self.num_experts * (imp * load).sum()
        else:
            self._lb_loss = None
        return original_output + final_lora_output

In [ ]:
class LBTrainer(Trainer):
    def __init__(self, *args, lb_coef: float = 1e-2, **kwargs):
        super().__init__(*args, **kwargs)
        self.lb_coef = lb_coef
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        loss, outputs = super().compute_loss(model, inputs, return_outputs=True, **kwargs)
        aux_loss = None
        for m in model.modules():
            if hasattr(m, "_lb_loss"):
                lb = getattr(m, "_lb_loss", None)
                if lb is not None:
                    aux_loss = lb if aux_loss is None else (aux_loss + lb)
        if aux_loss is not None and self.is_in_train:
            loss = loss + self.lb_coef * aux_loss
        return (loss, outputs) if return_outputs else loss

In [ ]:
def patch_roberta_with_edge_aware_lora(model, **kwargs):
    for layer in model.encoder.layer:
        layer.attention.self.query = EdgeAwareLoRALinear(layer.attention.self.query, **kwargs)
        layer.attention.self.value = EdgeAwareLoRALinear(layer.attention.self.value, **kwargs)
    return model

def compute_metrics(eval_pred):
    """ Calculates evaluation metrics for classification. """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

In [ ]:
print("Loading and preparing QNLI dataset...")
dataset = load_dataset("glue", "qnli")
tokenizer = RobertaTokenizer.from_pretrained("roberta-large")

def tokenize(example):
    return tokenizer(example["question"], example["sentence"], truncation=True, padding="max_length", max_length=512)

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

train_dataset = dataset["train"]
eval_dataset = dataset["validation"]
print("Dataset preparation finished.")

Loading and preparing QNLI dataset...
Dataset preparation finished.


In [ ]:
seeds = [42, 123, 7, 99, 101]
lb_coef = 1e-2
all_results = []

In [ ]:
for seed in seeds:
    print(f"\n{'='*40}\n  STARTING RUN: SEED={seed}, LB_COEF={lb_coef}\n{'='*40}\n")
    set_seed(seed)
    # --- CHANGE 3: Update output directories for QNLI ---
    out_dir = f"./ourlora_qnli_results/seed{seed}"
    os.makedirs(out_dir, exist_ok=True)

    # num_labels=2 is correct for QNLI (entailment or not)
    model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=2)
    model.roberta = patch_roberta_with_edge_aware_lora(model.roberta, num_experts=4, top_k=2, shared_rank=8, lora_alpha=16)

    trainable_modules = ["router", "lora_A_shared", "lora_B_shared", "lora_B_experts", "lora_A_experts"]
    for name, param in model.named_parameters():
        if not any(trainable_module in name for trainable_module in trainable_modules):
            param.requires_grad = False
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable Params: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")

    # --- CHANGE 4: Adjust training arguments for a larger dataset ---
    training_args = TrainingArguments(
        output_dir=f"./run_qnli_results/seed{seed}",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        learning_rate=2e-4,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=64,
        num_train_epochs=10,
        weight_decay=0.01,
        warmup_ratio=0.06,
        lr_scheduler_type="linear",
        logging_dir=f"./logs_qnli/seed{seed}",
        logging_steps=200, # Log less frequently
        report_to="none",
        fp16=True,
        seed=seed,
    )

    trainer = LBTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        lb_coef=lb_coef,
    )

    print(f"Starting training...")
    trainer.train()
    print(f"Training finished.")
    print(f"\n--- Analysis and Saving for SEED={seed} ---")

    final_metrics = trainer.evaluate()
    final_metrics['seed'] = seed
    all_results.append(final_metrics)


  STARTING RUN: SEED=42, LB_COEF=0.01



Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable Params: 1,081,344 / 356,443,138 (0.30%)
Starting training...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.746100,0.198377,0.925133
2,0.702600,0.181398,0.932272
3,0.692900,0.161334,0.939777
4,0.661700,0.163944,0.940326
5,0.658600,0.170427,0.941424
6,0.642400,0.176840,0.941424
7,0.627200,0.171828,0.942156
8,0.613200,0.172381,0.940326
9,0.612300,0.177365,0.943621
10,0.597200,0.183380,0.941973


Training finished.

--- Analysis and Saving for SEED=42 ---


...


  STARTING RUN: SEED=101, LB_COEF=0.01



Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable Params: 1,081,344 / 356,443,138 (0.30%)
Starting training...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.719300,0.168087,0.936848
2,0.683500,0.156063,0.942339
3,0.665500,0.150625,0.943987
4,0.647300,0.155020,0.944902
5,0.651500,0.149339,0.947282
6,0.629400,0.159596,0.944719
7,0.621400,0.168538,0.946366
8,0.615100,0.165915,0.947282
9,0.614000,0.174103,0.946916
10,0.593000,0.177348,0.946183


Training finished.

--- Analysis and Saving for SEED=101 ---


In [ ]:
import pandas as pd
import numpy as np

# Data extracted from the training logs
per_seed_data = [
    {'seed': 42, 'eval_accuracy': 0.941973, 'eval_loss': 0.183380},
    {'seed': 123, 'eval_accuracy': 0.944719, 'eval_loss': 0.178157},
    {'seed': 7, 'eval_accuracy': 0.940509, 'eval_loss': 0.179682},
    {'seed': 99, 'eval_accuracy': 0.945085, 'eval_loss': 0.174483},
    {'seed': 101, 'eval_accuracy': 0.946183, 'eval_loss': 0.177348}
]

# Create a DataFrame for easy viewing
results_df = pd.DataFrame(per_seed_data)

# Calculate summary statistics from the final epoch results
avg_accuracy = results_df['eval_accuracy'].mean()
best_final_epoch_accuracy = results_df['eval_accuracy'].max()
std_dev_accuracy = results_df['eval_accuracy'].std()

# This was the best accuracy found across any epoch of any seed
global_best_accuracy = 0.947282

# --- Print the Final Summary ---
print("--- Final Evaluation Results per Seed (from last epoch) ---")
print(results_df.to_string(index=False, float_format="%.4f"))

print("\n--- Summary Across 5 Seeds (based on final epoch) ---")
print(f"📊 Average Accuracy: {avg_accuracy:.4f}")
print(f"🏆 Best Final-Epoch Accuracy: {best_final_epoch_accuracy:.4f}")
print(f"🏆 Best Epoch Accuracy: {global_best_accuracy:.4f}")
print(f"📈 Std Deviation: {std_dev_accuracy:.4f}")
print(f"👑 Absolute Best Accuracy (any seed, any epoch): {global_best_accuracy:.4f}")

--- Final Evaluation Results per Seed (from last epoch) ---
 seed  eval_accuracy  eval_loss
   42         0.9420     0.1834
  123         0.9447     0.1782
    7         0.9405     0.1797
   99         0.9451     0.1745
  101         0.9462     0.1773

--- Summary Across 5 Seeds (based on final epoch) ---
📊 Average Accuracy: 0.9437
🏆 Best Final-Epoch Accuracy: 0.9462
🏆 Best Epoch Accuracy: 0.9473
📈 Std Deviation: 0.0024
👑 Absolute Best Accuracy (any seed, any epoch): 0.9473


# OurLoRA with aux for QQP roberta-large

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score
import os

# Enable memory-efficient Scaled Dot-Product Attention (SDPA) if using PyTorch 2.0+
if torch.cuda.is_available():
    try:
        import packaging.version as V
        if V.Version(torch.__version__) >= V.Version("2.0"):
            torch.backends.cuda.enable_mem_efficient_sdp(True)
            print("Enabled Scaled Dot-Product Attention (SDPA).")
    except Exception:
        pass

Enabled Scaled Dot-Product Attention (SDPA).


In [ ]:
class EdgeAwareLoRALinear(nn.Module):
    """
    Custom PyTorch module implementing a Mixture of Experts (MoE) LoRA.
    It replaces a standard linear layer, using a router to select expert pathways.
    """
    def __init__(self, original_layer, num_experts=4, top_k=2, shared_rank=4, expert_hidden_dim=32, lora_alpha=4, dropout=0.1):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.shared_rank = shared_rank
        self.expert_hidden_dim = expert_hidden_dim
        self.lora_alpha = lora_alpha
        self.scaling = self.lora_alpha / self.shared_rank
        self.original_layer = original_layer
        self._lb_loss = None

        in_features = original_layer.in_features
        out_features = original_layer.out_features

        self.router = nn.Linear(in_features, self.num_experts, bias=False)
        self.lora_A_shared = nn.Linear(in_features, self.shared_rank, bias=False)
        self.lora_B_experts = nn.ModuleDict()
        self.lora_A_experts = nn.ModuleDict()
        for i in range(self.num_experts):
            expert_name = str(i)
            self.lora_B_experts[expert_name] = nn.Linear(self.shared_rank, self.expert_hidden_dim, bias=False)
            self.lora_A_experts[expert_name] = nn.Linear(self.expert_hidden_dim, self.shared_rank, bias=False)
            nn.init.kaiming_uniform_(self.lora_B_experts[expert_name].weight, a=math.sqrt(5))
            nn.init.kaiming_uniform_(self.lora_A_experts[expert_name].weight, a=math.sqrt(5))
        self.lora_B_shared = nn.Linear(self.shared_rank, out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_A_shared.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B_shared.weight)
        self.lora_dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, sequence_length, hidden_dim = x.shape
        original_output = self.original_layer(x)
        x_flat = x.reshape(-1, hidden_dim)
        router_logits = self.router(x_flat)
        routing_weights_before = F.softmax(router_logits, dim=1, dtype=torch.float)
        routing_weights, selected_experts = torch.topk(routing_weights_before, self.top_k, dim=-1)
        routing_weights /= routing_weights.sum(dim=-1, keepdim=True)
        routing_weights = routing_weights.to(x_flat.dtype)
        expert_mask = F.one_hot(selected_experts, num_classes=self.num_experts).permute(2, 1, 0)
        x_lora = self.lora_A_shared(self.lora_dropout(x_flat))
        combined_expert_output = torch.zeros((batch_size * sequence_length, self.shared_rank), dtype=x_lora.dtype, device=x_lora.device)
        for expert_idx in range(self.num_experts):
            idx, top_x = torch.where(expert_mask[expert_idx])
            if top_x.shape[0] == 0:
                continue
            expert_input = x_lora[top_x]
            expert_output = self.lora_A_experts[str(expert_idx)](self.lora_B_experts[str(expert_idx)](expert_input))
            current_expert_output = expert_output * routing_weights[top_x, idx, None]
            combined_expert_output.index_add_(0, top_x, current_expert_output.to(x_lora.dtype))
        final_lora_output = self.lora_B_shared(combined_expert_output)
        final_lora_output = final_lora_output.reshape(batch_size, sequence_length, -1)
        final_lora_output = final_lora_output * self.scaling
        if self.training:
            P = routing_weights_before
            imp = P.mean(dim=0)
            with torch.no_grad():
                assign_counts = torch.bincount(selected_experts.reshape(-1), minlength=self.num_experts).float()
                load = assign_counts / assign_counts.sum().clamp_min(1.0)
            self._lb_loss = self.num_experts * (imp * load).sum()
        else:
            self._lb_loss = None
        return original_output + final_lora_output

In [ ]:
class LBTrainer(Trainer):
    def __init__(self, *args, lb_coef: float = 1e-2, **kwargs):
        super().__init__(*args, **kwargs)
        self.lb_coef = lb_coef
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        loss, outputs = super().compute_loss(model, inputs, return_outputs=True, **kwargs)
        aux_loss = None
        for m in model.modules():
            if hasattr(m, "_lb_loss"):
                lb = getattr(m, "_lb_loss", None)
                if lb is not None:
                    aux_loss = lb if aux_loss is None else (aux_loss + lb)
        if aux_loss is not None and self.is_in_train:
            loss = loss + self.lb_coef * aux_loss
        return (loss, outputs) if return_outputs else loss


In [ ]:
def patch_roberta_with_edge_aware_lora(model, **kwargs):
    for layer in model.encoder.layer:
        layer.attention.self.query = EdgeAwareLoRALinear(layer.attention.self.query, **kwargs)
        layer.attention.self.value = EdgeAwareLoRALinear(layer.attention.self.value, **kwargs)
    return model

# --- CHANGE 1: Update metrics to include F1 score for QQP ---
def compute_metrics(eval_pred):
    """ Calculates evaluation metrics for classification, including F1 score. """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1": f1}

In [ ]:
# --- CHANGE 2: Load the QQP dataset ---
print("Loading and preparing QQP dataset...")
dataset = load_dataset("glue", "qqp")
tokenizer = RobertaTokenizer.from_pretrained("roberta-large")

# --- CHANGE 3: Update tokenize function for question1/question2 pair ---
def tokenize(example):
    return tokenizer(example["question1"], example["question2"], truncation=True, padding="max_length", max_length=512)

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

train_dataset = dataset["train"]
eval_dataset = dataset["validation"]
print("Dataset preparation finished.")

Loading and preparing QQP dataset...


Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

Dataset preparation finished.


In [ ]:
seeds = [42, 123, 7, 99, 101]
lb_coef = 1e-2
all_results = []

In [ ]:
for seed in seeds:
    print(f"\n{'='*40}\n  STARTING RUN: SEED={seed}, LB_COEF={lb_coef}\n{'='*40}\n")
    set_seed(seed)
    # --- CHANGE 4: Update output directories for QQP ---
    out_dir = f"./ourlora_qqp_results/seed{seed}"
    os.makedirs(out_dir, exist_ok=True)

    # num_labels=2 is correct for QQP (duplicate or not)
    model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=2)
    model.roberta = patch_roberta_with_edge_aware_lora(model.roberta, num_experts=4, top_k=2, shared_rank=4, lora_alpha=4)

    trainable_modules = ["router", "lora_A_shared", "lora_B_shared", "lora_B_experts", "lora_A_experts"]
    for name, param in model.named_parameters():
        if not any(trainable_module in name for trainable_module in trainable_modules):
            param.requires_grad = False
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable Params: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")

    # --- CHANGE 5: Adjust training arguments for a very large dataset ---
    training_args = TrainingArguments(
        output_dir=f"./run_qqp_results/seed{seed}",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1", # F1 score is a key metric for QQP
        learning_rate=3e-4,
        per_device_train_batch_size=128,
        per_device_eval_batch_size=128,
        num_train_epochs=20,
        weight_decay=0.01,
        warmup_ratio=0.06,
        lr_scheduler_type="linear",
        logging_dir=f"./logs_qqp/seed{seed}",
        logging_steps=500, # Log much less frequently for a huge dataset
        report_to="none",
        fp16=True,
        seed=seed,
    )

    trainer = LBTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        lb_coef=lb_coef,
    )

    print(f"Starting training...")
    trainer.train()
    print(f"Training finished.")
    print(f"\n--- Analysis and Saving for SEED={seed} ---")

    final_metrics = trainer.evaluate()
    final_metrics['seed'] = seed
    all_results.append(final_metrics)


  STARTING RUN: SEED=42, LB_COEF=0.01



Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable Params: 638,976 / 356,000,770 (0.18%)
Starting training...


OutOfMemoryError: CUDA out of memory. Tried to allocate 128.00 MiB. GPU 0 has a total capacity of 23.51 GiB of which 10.75 MiB is free. Including non-PyTorch memory, this process has 23.38 GiB memory in use. Of the allocated memory 22.79 GiB is allocated by PyTorch, and 135.37 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
if all_results:
    results_df = pd.DataFrame(all_results)
    print("--- Final Evaluation Results per Seed ---")
    # --- CHANGE 6: Display F1 score in the summary table ---
    print(results_df[['seed', 'eval_accuracy', 'eval_f1', 'eval_loss']].to_string(float_format="%.4f"))

    avg_f1 = results_df['eval_f1'].mean()
    best_f1 = results_df['eval_f1'].max()
    std_dev_f1 = results_df['eval_f1'].std()

    avg_accuracy = results_df['eval_accuracy'].mean()
    best_accuracy = results_df['eval_accuracy'].max()

    print("\n--- Summary Across 5 Seeds on QQP ---")
    print(f"🎯 Average F1 Score: {avg_f1:.4f}")
    print(f"🏆 Best F1 Score:    {best_f1:.4f}")
    print(f"📈 Std Dev F1 Score: {std_dev_f1:.4f}")
    print("---")
    print(f"📊 Average Accuracy: {avg_accuracy:.4f}")
    print(f"✨ Best Accuracy:    {best_accuracy:.4f}")
else:
    print("No results to summarize.")

print("\nExperiment finished successfully! 🚀")

# Application of Edge Aware LoRA - MoE


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import os
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)
from datasets import load_dataset, Dataset
from sklearn.metrics import accuracy_score


class EdgeAwareLoRALinear(nn.Module):
    """
    Custom PyTorch module implementing a Mixture of Experts (MoE) LoRA.
    """
    def __init__(self, original_layer, num_experts=4, top_k=2, shared_rank=4, expert_hidden_dim=32, lora_alpha=4, dropout=0.1):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.shared_rank = shared_rank
        self.expert_hidden_dim = expert_hidden_dim
        self.lora_alpha = lora_alpha
        self.scaling = self.lora_alpha / self.shared_rank
        self.original_layer = original_layer
        self._lb_loss = None
        in_features = original_layer.in_features
        out_features = original_layer.out_features
        self.router = nn.Linear(in_features, self.num_experts, bias=False)
        self.lora_A_shared = nn.Linear(in_features, self.shared_rank, bias=False)
        self.lora_B_experts = nn.ModuleDict()
        self.lora_A_experts = nn.ModuleDict()
        for i in range(self.num_experts):
            expert_name = str(i)
            self.lora_B_experts[expert_name] = nn.Linear(self.shared_rank, self.expert_hidden_dim, bias=False)
            self.lora_A_experts[expert_name] = nn.Linear(self.expert_hidden_dim, self.shared_rank, bias=False)
            nn.init.kaiming_uniform_(self.lora_B_experts[expert_name].weight, a=math.sqrt(5))
            nn.init.kaiming_uniform_(self.lora_A_experts[expert_name].weight, a=math.sqrt(5))
        self.lora_B_shared = nn.Linear(self.shared_rank, out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_A_shared.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B_shared.weight)
        self.lora_dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, sequence_length, hidden_dim = x.shape
        original_output = self.original_layer(x)
        x_flat = x.reshape(-1, hidden_dim)
        router_logits = self.router(x_flat)
        routing_weights_before = F.softmax(router_logits, dim=1, dtype=torch.float)
        routing_weights, selected_experts = torch.topk(routing_weights_before, self.top_k, dim=-1)
        routing_weights /= routing_weights.sum(dim=-1, keepdim=True)
        routing_weights = routing_weights.to(x_flat.dtype)
        expert_mask = F.one_hot(selected_experts, num_classes=self.num_experts).permute(2, 1, 0)
        x_lora = self.lora_A_shared(self.lora_dropout(x_flat))
        combined_expert_output = torch.zeros((batch_size * sequence_length, self.shared_rank), dtype=x_lora.dtype, device=x_lora.device)
        for expert_idx in range(self.num_experts):
            idx, top_x = torch.where(expert_mask[expert_idx])
            if top_x.shape[0] == 0: continue
            expert_input = x_lora[top_x]
            expert_output = self.lora_A_experts[str(expert_idx)](self.lora_B_experts[str(expert_idx)](expert_input))
            current_expert_output = expert_output * routing_weights[top_x, idx, None]
            combined_expert_output.index_add_(0, top_x, current_expert_output.to(x_lora.dtype))
        final_lora_output = self.lora_B_shared(combined_expert_output)
        final_lora_output = final_lora_output.reshape(batch_size, sequence_length, -1)
        final_lora_output = final_lora_output * self.scaling
        if self.training:
            P = routing_weights_before
            imp = P.mean(dim=0)
            with torch.no_grad():
                assign_counts = torch.bincount(selected_experts.reshape(-1), minlength=self.num_experts).float()
                load = assign_counts / assign_counts.sum().clamp_min(1.0)
            self._lb_loss = self.num_experts * (imp * load).sum()
        else:
            self._lb_loss = None
        return original_output + final_lora_output

class LBTrainer(Trainer):
    def __init__(self, *args, lb_coef: float = 1e-2, **kwargs):
        super().__init__(*args, **kwargs)
        self.lb_coef = lb_coef
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        loss, outputs = super().compute_loss(model, inputs, return_outputs=True, **kwargs)
        aux_loss = None
        for m in model.modules():
            if hasattr(m, "_lb_loss"):
                lb = getattr(m, "_lb_loss", None)
                if lb is not None:
                    aux_loss = lb if aux_loss is None else (aux_loss + lb)
        if aux_loss is not None and self.is_in_train:
            loss = loss + self.lb_coef * aux_loss
        return (loss, outputs) if return_outputs else loss


def patch_roberta_with_edge_aware_lora(model, **kwargs):
    for layer in model.encoder.layer:
        layer.attention.self.query = EdgeAwareLoRALinear(layer.attention.self.query, **kwargs)
        layer.attention.self.value = EdgeAwareLoRALinear(layer.attention.self.value, **kwargs) # <-- FIX WAS HERE
    return model

# --- Setup ---
set_seed(42)
MODEL_NAME = "roberta-base"
LABELS = ["NEGATIVE", "POSITIVE"]
tricky_sentence = "This movie was not bad at all."

# --- Helper function to make predictions ---
def predict(model, tokenizer, text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
    pred_idx = torch.argmax(logits, dim=-1).item()
    return LABELS[pred_idx]

# --- Step 1: Show the "General" Model Failing ---
print("--- 1. Testing General Model (roberta-base) ---")
general_tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
general_model = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to("cuda")
general_model.eval()

total_params = sum(p.numel() for p in general_model.parameters())
print(f"Total parameters in base model: {total_params:,}")

pred_before = predict(general_model, general_tokenizer, tricky_sentence)
print(f"Sentence: '{tricky_sentence}'")
print(f"Prediction BEFORE personalization: {pred_before}")

# --- Step 2: Define "Personalization" Data ---
print("\n--- 2. Creating tiny 'personal' dataset ---")
personal_data = {
    'sentence': [
        "This was not bad.", "That was not terrible.", "I'm not unhappy.",
        "It's not the worst.", "That's pretty good.", "I liked it a lot.",
        "This was not bad at all.", "Not a terrible movie.", "He isn't wrong.",
        "This is a great film.", "The service was not bad.", "The food was not terrible."
    ],
    'labels': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1] # All are POSITIVE
}
personal_dataset = Dataset.from_dict(personal_data)

def tokenize(example):
    return general_tokenizer(example["sentence"], truncation=True, padding="max_length", max_length=64)
personal_dataset = personal_dataset.map(tokenize, batched=True, remove_columns=["sentence"])
personal_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# --- Step 3: Patch the Model with Edge-Aware LoRA ---
print("\n--- 3. Patching model with Edge-Aware LoRA ---")
personal_model = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
personal_model.roberta = patch_roberta_with_edge_aware_lora(
    personal_model.roberta, num_experts=2, top_k=1, shared_rank=4, expert_hidden_dim=16
)
for name, param in personal_model.named_parameters():
    if not any(x in name for x in ["router", "lora_A_shared", "lora_B_shared", "lora_B_experts", "lora_A_experts"]):
        param.requires_grad = False
    else:
        param.requires_grad = True

print("Model patched. Calculating parameter breakdown...")

# --- Step 4: Rapid "On-Device" Fine-Tuning ---
print("\n--- 4. Starting rapid personalization (fine-tuning)... ---")
training_args = TrainingArguments(
    output_dir="./personalization-demo",
    per_device_train_batch_size=2,
    num_train_epochs=50,
    learning_rate=1e-4,
    logging_steps=10,
    report_to="none",
    fp16=True,
    seed=42
)

trainer = LBTrainer(
    model=personal_model,
    args=training_args,
    train_dataset=personal_dataset
)

trainer.train()
print("--- Personalization (fine-tuning) COMPLETE ---")

# --- Step 5: Show the "Personalized" Model Succeeding ---
print("\n--- 5. Testing Personalized Model ---")
personal_model.to("cuda")
personal_model.eval()

pred_after = predict(personal_model, general_tokenizer, tricky_sentence)
print(f"Sentence: '{tricky_sentence}'")
print(f"Prediction AFTER personalization: {pred_after}")


print("\n--- 6. Efficiency Analysis ---")
# --- Split trainable params into Edge vs Cloud ---
edge_only_params = 0
cloud_trainable_params = 0
edge_state_dict = {}

for name, param in personal_model.named_parameters():
    if param.requires_grad:
        # According to your design, router and experts are on the edge
        if "router" in name or "lora_A_experts" in name or "lora_B_experts" in name:
            edge_only_params += param.numel()
            edge_state_dict[name] = param.data
        else:
            # A_shared and B_shared are the "Cloud" components
            cloud_trainable_params += param.numel()

total_trainable_params = edge_only_params + cloud_trainable_params

# --- Calculate percentages ---
edge_percent_of_total = (edge_only_params / total_params) * 100
cloud_percent_of_total = (cloud_trainable_params / total_params) * 100
total_trainable_percent = (total_trainable_params / total_params) * 100

# --- Get file sizes ---
full_model_path = "full_base_model.pth"
torch.save(general_model.state_dict(), full_model_path)
full_model_size_mb = os.path.getsize(full_model_path) / (1024 * 1024)

edge_weights_path = "edge_weights_ONLY.pth"
torch.save(edge_state_dict, edge_weights_path)
edge_weights_size_mb = os.path.getsize(edge_weights_path) / (1024 * 1024)

print("\n" + "="*52)
print("     🚀 PERSONALIZE-ON-EDGE DEMO (roberta-base) 🚀")
print("="*52)

print("\n--- PERFORMANCE ---")
print(f"Sentence:            '{tricky_sentence}'")
print(f"General Model:        {pred_before}")
print(f"Personalized Model:   {pred_after}")

print("\n--- EFFICIENCY ---")
print(f"Total Base Parameters:   {total_params:,}")

print("\n--- PARAMETER BREAKDOWN (Our LoRA-MoE) ---")
print(f"TOTAL Trainable:     {total_trainable_params:,} ({total_trainable_percent:.3f}% of total)")
print("--------------------------------------------------")
print(f"  > On-Device 'Edge':  {edge_only_params:,} ({edge_percent_of_total:.4f}%)")
print(f"  > 'Cloud' (Shared):  {cloud_trainable_params:,} ({cloud_percent_of_total:.4f}%)")

print("\n--- STORAGE FOOTPRINT ---")
print(f"Full Model Storage:      {full_model_size_mb:.2f} MB")
print(f"ON-DEVICE Storage:       {edge_weights_size_mb:.2f} MB   <-- (This is what the user stores!)")
print("="*52)

if pred_before != pred_after:
    print("\n The model adapted to the user's personal style.")

# Clean up the large files
os.remove(full_model_path)
os.remove(edge_weights_path)

--- 1. Testing General Model (roberta-base) ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...


--- 3. Patching model with Edge-Aware LoRA ---


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model patched. Calculating parameter breakdown...

--- 4. Starting rapid personalization (fine-tuning)... ---


Step,Training Loss
10,1.052700
20,1.013900
30,1.031500
40,1.012000
50,1.003900
60,1.025900
70,0.991400
80,1.015300
90,1.001200
100,1.022600


--- Personalization (fine-tuning) COMPLETE ---

--- 5. Testing Personalized Model ---
Sentence: 'This movie was not bad at all.'
Prediction AFTER personalization: POSITIVE

--- 6. Efficiency Analysis ---

     🚀 PERSONALIZE-ON-EDGE DEMO (roberta-base) 🚀

--- PERFORMANCE ---
Sentence:            'This movie was not bad at all.'
General Model:        NEGATIVE
Personalized Model:   POSITIVE

--- EFFICIENCY ---
Total Base Parameters:   124,647,170

--- PARAMETER BREAKDOWN (Our LoRA-MoE) ---
TOTAL Trainable:     190,464 (0.153% of total)
--------------------------------------------------
  > On-Device 'Edge':  43,008 (0.0345%)
  > 'Cloud' (Shared):  147,456 (0.1183%)

--- STORAGE FOOTPRINT ---
Full Model Storage:      475.57 MB
ON-DEVICE Storage:       0.21 MB   <-- (This is what the user stores!)

 The model adapted to the user's personal style.
